<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/04_Blocking_rebuild.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Rebuild Blocking Indexes (condensed)

This replaces the long exploratory version. It rebuilds exactly the 5 production
index variables (`name_index`, `token_index`, `address_token_index`, `char3_index`,
`rare_two_token_index_10`) plus the `missed_s2_ids` set they depend on, with no
experiments/threshold sweeps/transliteration analysis in between. Results are the
same as before — the ≤10 rare two-token-pair blocker gives ~91.4666% blocking recall.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "test"
)

BLOCKING_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(BLOCKING_OUTPUT_ROOT, exist_ok=True)

print("Cleaned data:")
print(CLEANED_DATA_ROOT)

print("\nBlocking output:")
print(BLOCKING_OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cleaned data:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Blocking output:
/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs


In [ ]:
RAW_S1_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source1.tsv"
)

RAW_S2_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source2.tsv"
)
RAW_S3_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source3.tsv"
)



In [ ]:
GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv")

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype={
        "source1_entity_id": "string",
        "matched_entity_ids": "string"
    }
)

print("Ground truth rows:", len(gt))


Ground truth rows: 2206821


In [ ]:
gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground truth lookup entries:", len(gt_lookup))


Ground truth lookup entries: 2206821


In [ ]:
s1_blocking = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core",
        "address_norm"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string",
        "address_norm": "string"
    }
)

print(s1_blocking.shape)


(2206821, 4)


In [ ]:
NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    tokens = str(name).split()

    return [
        token
        for token in tokens
        if token not in NAME_STOPWORDS and len(token) >= 2
    ]


### 1. Exact country + name index (`name_index`)

In [ ]:
from collections import defaultdict

def build_name_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):
        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

print("Building S2 exact-name index...")
name_index = build_name_index(S2_PATH)
print("name_index:", len(name_index))


Building S2 exact-name index...
name_index: 3597772


### 2. Name-token index (`token_index`)

In [ ]:
from collections import Counter

name_token_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        name_token_counter.update(set(meaningful_tokens(name)))

    del chunk
    gc.collect()

allowed_name_tokens = {
    token
    for token, freq in name_token_counter.items()
    if 2 <= freq <= 500
}

print("Unique name tokens:", len(name_token_counter))
print("Allowed name tokens:", len(allowed_name_tokens))

del name_token_counter
gc.collect()


Unique name tokens: 810209
Allowed name tokens: 167295


0

In [ ]:
token_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    for entity_id, name in zip(chunk["entity_id"], chunk["name_core"]):
        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("token_index:", len(token_index))


token_index: 167295


### 3. Address-token index (`address_token_index`)

In [ ]:
address_token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["address_norm"],
    chunksize=200_000,
    dtype={"address_norm": "string"}
):
    for address in chunk["address_norm"].dropna():
        for token in set(str(address).split()):
            if token:
                address_token_counts[token] += 1
    del chunk
    gc.collect()

MIN_ADDRESS_TOKEN_FREQ = 2
MAX_ADDRESS_TOKEN_FREQ = 500

allowed_address_tokens = {
    token
    for token, count in address_token_counts.items()
    if MIN_ADDRESS_TOKEN_FREQ <= count <= MAX_ADDRESS_TOKEN_FREQ
}

print("Unique address tokens:", len(address_token_counts))
print("Allowed address tokens:", len(allowed_address_tokens))

del address_token_counts
gc.collect()


Unique address tokens: 607900
Allowed address tokens: 404250


0

In [ ]:
address_token_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "address_norm"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "address_norm": "string"
    }
):
    for entity_id, address in zip(chunk["entity_id"], chunk["address_norm"]):
        if pd.isna(address):
            continue

        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                address_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("address_token_index:", len(address_token_index))


address_token_index: 404250


### 4. `missed_s2_ids` — true S2 matches not caught by the 3 blockers above

In [ ]:
missed_s2_ids = set()

total_true = 0
captured_existing = 0

for row in s1_blocking.itertuples(index=False):
    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    true_ids = {
        x
        for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    }

    if not true_ids:
        continue

    total_true += len(true_ids)

    candidate_set = set()

    if pd.notna(name) and name != "":
        candidate_set.update(name_index.get((country, name), []))

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                candidate_set.update(token_index.get(token, []))

    if pd.notna(address) and address != "":
        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                candidate_set.update(address_token_index.get(token, []))

    captured = true_ids.intersection(candidate_set)

    captured_existing += len(captured)

    missed_s2_ids.update(true_ids - captured)

print("Total true S2 matches:", total_true)
print("Captured by existing blocker:", captured_existing)
print("Missed by existing blocker:", len(missed_s2_ids))


Total true S2 matches: 3693619
Captured by existing blocker: 3222405
Missed by existing blocker: 471214


### 5. Character 3-gram index (`char3_index`) — conservative version, freq 2–500

In [ ]:
def char_ngrams(text, n=3):
    if pd.isna(text):
        return set()

    text = str(text).replace(" ", "")

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }

char3_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        char3_counter.update(char_ngrams(name, n=3))
    del chunk
    gc.collect()

allowed_char3 = {
    gram
    for gram, freq in char3_counter.items()
    if 2 <= freq <= 500
}

print("Unique 3-grams:", len(char3_counter))
print("Allowed 3-grams:", len(allowed_char3))

del char3_counter
gc.collect()


In [ ]:
char3_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    for entity_id, name in zip(chunk["entity_id"], chunk["name_core"]):
        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):
            if gram in allowed_char3:
                char3_index[gram].append(entity_id)

    del chunk
    gc.collect()

print("char3_index:", len(char3_index))


char3_index: 65473


### 6. Rare two-token-pair index (`rare_two_token_index_10`), freq ≤ 10

In [ ]:
missed_s2_names = {}

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    mask = chunk["entity_id"].isin(missed_s2_ids)

    for entity_id, name in zip(
        chunk.loc[mask, "entity_id"],
        chunk.loc[mask, "name_core"]
    ):
        missed_s2_names[entity_id] = name

    del chunk
    gc.collect()

print("Missed S2 names:", len(missed_s2_names))


In [ ]:
from itertools import combinations

def name_token_pairs(name):
    tokens = sorted(set(meaningful_tokens(name)))

    if len(tokens) < 2:
        return set()

    return set(combinations(tokens, 2))

two_token_frequency = defaultdict(int)

for name in missed_s2_names.values():
    if pd.isna(name):
        continue
    for pair in name_token_pairs(name):
        two_token_frequency[pair] += 1

print("Unique two-token pairs:", len(two_token_frequency))


In [ ]:
rare_two_token_index_10 = defaultdict(list)

for s2_id, name in missed_s2_names.items():
    if pd.isna(name):
        continue

    for pair in name_token_pairs(name):
        if two_token_frequency[pair] <= 10:
            rare_two_token_index_10[pair].append(s2_id)

print("rare_two_token_index_10:", len(rare_two_token_index_10))
print(
    "Indexed references:",
    sum(len(v) for v in rare_two_token_index_10.values())
)


rare_two_token_index_10: 353966
Indexed references: 522162


### Verify everything

In [ ]:
print("name_index:", len(name_index))
print("token_index:", len(token_index))
print("address_token_index:", len(address_token_index))
print("char3_index:", len(char3_index))
print("rare_two_token_index_10:", len(rare_two_token_index_10))


In [ ]:
import os
import gc
import pandas as pd

CANDIDATE_OUTPUT_DIR = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(CANDIDATE_OUTPUT_DIR, exist_ok=True)

CANDIDATE_PATH = os.path.join(
    CANDIDATE_OUTPUT_DIR,
    "train_candidate_pairs.tsv"
)

# Remove an old partial output if it exists
if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print("Output:", CANDIDATE_PATH)

Output: /content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/train_candidate_pairs.tsv


In [ ]:
# =========================================================
# BUILD SELECTIVE NAME-TOKEN INDEX
# =========================================================

NAME_TOKEN_MAX_CANDIDATES = 100

selective_token_index = {}

for token, ids in token_index.items():

    if len(ids) <= NAME_TOKEN_MAX_CANDIDATES:
        selective_token_index[token] = ids

print("Original name-token keys:", len(token_index))
print("Selective name-token keys:", len(selective_token_index))

print(
    "Original indexed references:",
    sum(len(v) for v in token_index.values())
)

print(
    "Selective indexed references:",
    sum(len(v) for v in selective_token_index.values())
)

Original name-token keys: 167295
Selective name-token keys: 161071
Original indexed references: 3230167
Selective indexed references: 1999717


In [ ]:
# =========================================================
# BUILD SELECTIVE ADDRESS-TOKEN INDEX
# =========================================================

ADDRESS_TOKEN_MAX_CANDIDATES = 100

selective_address_token_index = {}

for token, ids in address_token_index.items():

    if len(ids) <= ADDRESS_TOKEN_MAX_CANDIDATES:
        selective_address_token_index[token] = ids

print("Original address-token keys:", len(address_token_index))
print(
    "Selective address-token keys:",
    len(selective_address_token_index)
)

print(
    "Original indexed references:",
    sum(len(v) for v in address_token_index.values())
)

print(
    "Selective indexed references:",
    sum(
        len(v)
        for v in selective_address_token_index.values()
    )
)

Original address-token keys: 404250
Selective address-token keys: 389029
Original indexed references: 6812664
Selective indexed references: 3568962


In [ ]:
# =========================================================
# TEST CANDIDATE VOLUME ON 50K S1
# =========================================================

TEST_ROWS = 50_000

test_s1 = s1_blocking.iloc[:TEST_ROWS]

candidate_counts = []

for row in test_s1.itertuples(index=False):

    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    candidates = set()

    # 1. Exact country + name
    if pd.notna(country) and pd.notna(name):
        candidates.update(
            name_index.get((country, name), [])
        )

    # 2. Selective name token
    if pd.notna(name):

        for token in meaningful_tokens(name):

            candidates.update(
                selective_token_index.get(token, [])
            )

    # 3. Selective address token
    if pd.notna(address):

        for token in set(str(address).split()):

            candidates.update(
                selective_address_token_index.get(token, [])
            )

    # 4. Conservative 3-gram
    if pd.notna(name):

        for gram in char_ngrams(name, n=3):

            candidates.update(
                char3_index.get(gram, [])
            )

    # 5. Rare two-token pair <=10
    if pd.notna(name):

        for pair in name_token_pairs(name):

            candidates.update(
                rare_two_token_index_10.get(pair, [])
            )

    candidate_counts.append(len(candidates))

candidate_counts = pd.Series(candidate_counts)

print("====================================")
print("50K CANDIDATE VOLUME TEST")
print("====================================")
print("Mean:", candidate_counts.mean())
print("Median:", candidate_counts.median())
print("P75:", candidate_counts.quantile(0.75))
print("P90:", candidate_counts.quantile(0.90))
print("P95:", candidate_counts.quantile(0.95))
print("P99:", candidate_counts.quantile(0.99))
print("Max:", candidate_counts.max())
print("Total:", candidate_counts.sum())
print(
    "Estimated 2.2M candidates:",
    int(
        candidate_counts.mean()
        * len(s1_blocking)
    )
)

50K CANDIDATE VOLUME TEST
Mean: 98.1712
Median: 48.0
P75: 104.0
P90: 279.0
P95: 428.0
P99: 714.0
Max: 1944
Total: 4908560
Estimated 2.2M candidates: 216646265


In [ ]:
# =========================================================
# S3 BASELINE: COUNTRY + EXACT NAME
# =========================================================

s3_name_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "country_norm", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
):

    for entity_id, country, name in zip(
        chunk["entity_id"],
        chunk["country_norm"],
        chunk["name_core"]
    ):

        if pd.isna(country) or pd.isna(name):
            continue

        s3_name_index[(country, name)].append(entity_id)

    del chunk
    gc.collect()

print("S3 name index:", len(s3_name_index))

S3 name index: 3850257


In [ ]:
# =========================================================
# FINAL SELECTIVE S2 NAME-TOKEN INDEX
# =========================================================

S2_NAME_MAX_FREQ = 10

s2_selective_name_index = {
    token: ids
    for token, ids in token_index.items()
    if len(ids) <= S2_NAME_MAX_FREQ
}

print("S2 selective name keys:", len(s2_selective_name_index))
print(
    "S2 selective references:",
    sum(len(v) for v in s2_selective_name_index.values())
)

In [ ]:
# =========================================================
# BUILD FINAL S3 BLOCKING INDEXES
# =========================================================

from collections import defaultdict
import pandas as pd
import gc

S3_NAME_MAX_FREQ = 10

# ---------------------------------------------------------
# A. S3 exact country + name
# ---------------------------------------------------------

s3_name_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "country_norm", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
):

    for entity_id, country, name in zip(
        chunk["entity_id"],
        chunk["country_norm"],
        chunk["name_core"]
    ):
        if pd.isna(country) or pd.isna(name):
            continue

        s3_name_index[(country, name)].append(entity_id)

    del chunk
    gc.collect()

print("S3 exact-name keys:", len(s3_name_index))


# ---------------------------------------------------------
# B. S3 name-token frequency
# ---------------------------------------------------------

s3_token_freq = defaultdict(int)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):

    for name in chunk["name_core"]:

        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):
            s3_token_freq[token] += 1

    del chunk
    gc.collect()

s3_allowed_tokens = {
    token
    for token, freq in s3_token_freq.items()
    if 2 <= freq <= S3_NAME_MAX_FREQ
}

print("S3 total tokens:", len(s3_token_freq))
print("S3 selective tokens:", len(s3_allowed_tokens))

del s3_token_freq
gc.collect()


# ---------------------------------------------------------
# C. S3 selective name-token index
# ---------------------------------------------------------

s3_token_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):

            if token in s3_allowed_tokens:
                s3_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("S3 selective token keys:", len(s3_token_index))


# ---------------------------------------------------------
# D. S3 3-gram frequencies
# ---------------------------------------------------------

s3_char3_freq = defaultdict(int)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):

    for name in chunk["name_core"]:

        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):
            s3_char3_freq[gram] += 1

    del chunk
    gc.collect()

s3_allowed_char3 = {
    gram
    for gram, freq in s3_char3_freq.items()
    if 2 <= freq <= 500
}

print("S3 allowed 3-grams:", len(s3_allowed_char3))

del s3_char3_freq
gc.collect()


# ---------------------------------------------------------
# E. S3 3-gram index
# ---------------------------------------------------------

s3_char3_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):

            if gram in s3_allowed_char3:
                s3_char3_index[gram].append(entity_id)

    del chunk
    gc.collect()

print("S3 3-gram keys:", len(s3_char3_index))

print("\nS3 indexes ready.")

In [ ]:
# =========================================================
# BUILD S3 RARE TWO-TOKEN-PAIR INDEX
# =========================================================

s3_pair_freq = defaultdict(int)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for name in chunk["name_core"]:

        if pd.isna(name):
            continue

        for pair in name_token_pairs(name):
            s3_pair_freq[pair] += 1

    del chunk
    gc.collect()

print("S3 unique token pairs:", len(s3_pair_freq))


s3_rare_pair_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for pair in name_token_pairs(name):

            if s3_pair_freq[pair] <= 10:
                s3_rare_pair_index[pair].append(entity_id)

    del chunk
    gc.collect()

print("S3 rare pair keys:", len(s3_rare_pair_index))
print(
    "S3 rare pair references:",
    sum(len(v) for v in s3_rare_pair_index.values())
)

del s3_pair_freq
gc.collect()

In [30]:
import os

CANDIDATE_PATH = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs",
    "candidate_pairs.tsv"
)

if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print(CANDIDATE_PATH)

/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv


In [31]:
LOCAL_CANDIDATE_PATH = "/content/candidate_pairs.tsv"

if os.path.exists(LOCAL_CANDIDATE_PATH):
    os.remove(LOCAL_CANDIDATE_PATH)

print(LOCAL_CANDIDATE_PATH)

/content/candidate_pairs.tsv


In [32]:
# =========================================================
# 10K CANDIDATE VOLUME TEST — SIMPLIFIED
# =========================================================

TEST_ROWS = 10_000

test_s1 = s1_blocking.iloc[:TEST_ROWS]

s2_counts = []
s3_counts = []
total_counts = []

for row in test_s1.itertuples(index=False):

    name = row.name_core
    country = row.country_norm

    # =====================================================
    # S2
    # =====================================================

    s2_candidates = set()

    # Exact country + name
    if pd.notna(country) and pd.notna(name):
        s2_candidates.update(
            name_index.get(
                (country, name),
                []
            )
        )

    # Rare 2-token pair <=10
    if pd.notna(name):

        for pair in name_token_pairs(name):
            s2_candidates.update(
                rare_two_token_index_10.get(pair, [])
            )

    # =====================================================
    # S3
    # =====================================================

    s3_candidates = set()

    # Exact country + name
    if pd.notna(country) and pd.notna(name):
        s3_candidates.update(
            s3_name_index.get(
                (country, name),
                []
            )
        )

    # Selective name tokens <=10
    if pd.notna(name):

        for token in meaningful_tokens(name):

            ids = s3_token_index.get(token)

            if ids:
                s3_candidates.update(ids)

    # =====================================================
    # Record statistics
    # =====================================================

    s2_counts.append(len(s2_candidates))
    s3_counts.append(len(s3_candidates))
    total_counts.append(
        len(s2_candidates) +
        len(s3_candidates)
    )


s2_counts = pd.Series(s2_counts)
s3_counts = pd.Series(s3_counts)
total_counts = pd.Series(total_counts)

print("========================================")
print("SIMPLIFIED 10K CANDIDATE TEST")
print("========================================")

print("\nS2")
print("Mean:", s2_counts.mean())
print("Median:", s2_counts.median())
print("P90:", s2_counts.quantile(.90))
print("P95:", s2_counts.quantile(.95))
print("P99:", s2_counts.quantile(.99))
print("Max:", s2_counts.max())
print("Total:", s2_counts.sum())

print("\nS3")
print("Mean:", s3_counts.mean())
print("Median:", s3_counts.median())
print("P90:", s3_counts.quantile(.90))
print("P95:", s3_counts.quantile(.95))
print("P99:", s3_counts.quantile(.99))
print("Max:", s3_counts.max())
print("Total:", s3_counts.sum())

print("\nCOMBINED")
print("Mean:", total_counts.mean())
print("Median:", total_counts.median())
print("P90:", total_counts.quantile(.90))
print("P95:", total_counts.quantile(.95))
print("P99:", total_counts.quantile(.99))
print("Max:", total_counts.max())
print("Total:", total_counts.sum())

estimated_total = (
    total_counts.mean()
    * len(s1_blocking)
)

print(
    "\nEstimated full candidate pairs:",
    f"{estimated_total:,.0f}"
)

SIMPLIFIED 10K CANDIDATE TEST

S2
Mean: 16.8712
Median: 3.0
P90: 30.0
P95: 75.0
P99: 262.0100000000002
Max: 647
Total: 168712

S3
Mean: 16.6818
Median: 2.0
P90: 38.0
P95: 71.0
P99: 259.0100000000002
Max: 584
Total: 166818

COMBINED
Mean: 33.553
Median: 5.0
P90: 65.0
P95: 145.04999999999927
P99: 513.0
Max: 1231
Total: 335530

Estimated full candidate pairs: 74,045,465


In [33]:
import os

if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print("Partial Drive candidate file removed.")

Partial Drive candidate file removed.


In [34]:
MAX_CANDIDATES_PER_SOURCE = 50

s2_counts = []
s3_counts = []

for row in test_s1.itertuples(index=False):

    name = row.name_core
    country = row.country_norm

    # -----------------------------
    # S2
    # -----------------------------
    s2_candidates = set()

    if pd.notna(country) and pd.notna(name):
        s2_candidates.update(
            name_index.get((country, name), [])
        )

    if pd.notna(name):
        for pair in name_token_pairs(name):
            s2_candidates.update(
                rare_two_token_index_10.get(pair, [])
            )

    if len(s2_candidates) > MAX_CANDIDATES_PER_SOURCE:
        s2_candidates = set(
            list(s2_candidates)[:MAX_CANDIDATES_PER_SOURCE]
        )

    # -----------------------------
    # S3
    # -----------------------------
    s3_candidates = set()

    if pd.notna(country) and pd.notna(name):
        s3_candidates.update(
            s3_name_index.get((country, name), [])
        )

    if pd.notna(name):
        for token in meaningful_tokens(name):
            ids = s3_token_index.get(token)
            if ids:
                s3_candidates.update(ids)

    if len(s3_candidates) > MAX_CANDIDATES_PER_SOURCE:
        s3_candidates = set(
            list(s3_candidates)[:MAX_CANDIDATES_PER_SOURCE]
        )

    s2_counts.append(len(s2_candidates))
    s3_counts.append(len(s3_candidates))


s2_counts = pd.Series(s2_counts)
s3_counts = pd.Series(s3_counts)

print("S2 mean:", s2_counts.mean())
print("S2 P95:", s2_counts.quantile(.95))
print("S2 max:", s2_counts.max())

print("S3 mean:", s3_counts.mean())
print("S3 P95:", s3_counts.quantile(.95))
print("S3 max:", s3_counts.max())

print(
    "Estimated combined pairs:",
    int(
        (s2_counts.mean() + s3_counts.mean())
        * len(s1_blocking)
    )
)

S2 mean: 9.755
S2 P95: 50.0
S2 max: 50
S3 mean: 9.7391
S3 P95: 50.0
S3 max: 50
Estimated combined pairs: 43019989


In [ ]:
# =========================================================
# FINAL STREAMLINED CANDIDATE GENERATOR
# S2 + S3
# =========================================================

import os
import gc
import pandas as pd

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

LOCAL_CANDIDATE_PATH = "/content/candidate_pairs.tsv"

if os.path.exists(LOCAL_CANDIDATE_PATH):
    os.remove(LOCAL_CANDIDATE_PATH)

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------

CHUNK_SIZE = 25_000

first_write = True
total_pairs = 0
total_s1 = 0


# =========================================================
# PROCESS S1 IN CHUNKS
# =========================================================

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S1_PATH,
        sep="\t",
        usecols=[
            "entity_id",
            "country_norm",
            "name_core"
        ],
        chunksize=CHUNK_SIZE,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ),
    start=1
):

    rows = []

    # -----------------------------------------------------
    # Process every S1 entity
    # -----------------------------------------------------

    for row in chunk.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core

        # =================================================
        # S2 CANDIDATES
        # =================================================

        s2_candidates = set()

        # 1. Exact country + name
        if pd.notna(country) and pd.notna(name):

            s2_candidates.update(
                name_index.get(
                    (country, name),
                    []
                )
            )

        # 2. Rare two-token pair <= 10
        if pd.notna(name):

            for pair in name_token_pairs(name):

                ids = rare_two_token_index_10.get(pair)

                if ids:
                    s2_candidates.update(ids)

        # Write S2 candidates
        for s2_id in s2_candidates:

            rows.append(
                (
                    s1_id,
                    s2_id,
                    "S2"
                )
            )

        # =================================================
        # S3 CANDIDATES
        # =================================================

        s3_candidates = set()

        # 1. Exact country + name
        if pd.notna(country) and pd.notna(name):

            s3_candidates.update(
                s3_name_index.get(
                    (country, name),
                    []
                )
            )

        # 2. Selective name tokens
        if pd.notna(name):

            for token in meaningful_tokens(name):

                ids = s3_token_index.get(token)

                if ids:
                    s3_candidates.update(ids)

        # Write S3 candidates
        for s3_id in s3_candidates:

            rows.append(
                (
                    s1_id,
                    s3_id,
                    "S3"
                )
            )

    # -----------------------------------------------------
    # Convert current chunk to DataFrame
    # -----------------------------------------------------

    if rows:

        out = pd.DataFrame(
            rows,
            columns=[
                "source1_entity_id",
                "candidate_entity_id",
                "source"
            ]
        )

        # Remove duplicate pairs inside this chunk
        out.drop_duplicates(
            inplace=True
        )

        # -------------------------------------------------
        # Write locally
        # -------------------------------------------------

        out.to_csv(
            LOCAL_CANDIDATE_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        total_pairs += len(out)

        del out
        del rows

    total_s1 += len(chunk)

    # -----------------------------------------------------
    # Progress
    # -----------------------------------------------------

    if chunk_no % 4 == 0:

        print(
            f"S1 processed: {total_s1:,} | "
            f"Candidate pairs: {total_pairs:,}"
        )

    del chunk
    gc.collect()

    first_write = False


# =========================================================
# COMPLETE
# =========================================================

print("\n========================================")
print("CANDIDATE GENERATION COMPLETE")
print("========================================")
print(
    "S1 processed:",
    f"{total_s1:,}"
)
print(
    "Candidate pairs:",
    f"{total_pairs:,}"
)
print(
    "Local output:",
    LOCAL_CANDIDATE_PATH
)

In [ ]:
import os

size_gb = os.path.getsize(
    LOCAL_CANDIDATE_PATH
) / (1024**3)

print(f"File size: {size_gb:.2f} GB")

In [35]:
FINAL_DRIVE_PATH = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs",
    "candidate_pairs.tsv"
)

import shutil

shutil.copy2(
    LOCAL_CANDIDATE_PATH,
    FINAL_DRIVE_PATH
)

print("Saved to:")
print(FINAL_DRIVE_PATH)

FileNotFoundError: [Errno 2] No such file or directory: '/content/candidate_pairs.tsv'

In [36]:
import pandas as pd

CANDIDATE_PATH = (
    "/content/drive/MyDrive/Amazon ML Challenge 2026/"
    "05_Outputs/Candidate_Pairs/candidate_pairs.tsv"
)

s2_count = 0
s3_count = 0
total = 0

for chunk in pd.read_csv(
    CANDIDATE_PATH,
    sep="\t",
    usecols=["source"],
    chunksize=1_000_000
):
    counts = chunk["source"].value_counts()

    s2_count += counts.get("S2", 0)
    s3_count += counts.get("S3", 0)

    total += len(chunk)

    print(
        f"Read {total:,} candidates | "
        f"S2: {s2_count:,} | "
        f"S3: {s3_count:,}"
    )

print("\n==============================")
print("FINAL CANDIDATE COUNTS")
print("==============================")
print("Total:", f"{total:,}")
print("S2:", f"{s2_count:,}")
print("S3:", f"{s3_count:,}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv'

In [37]:
import glob
import os

for pattern in ["**/test_source1*.tsv", "**/test_source2*.tsv", "**/test_source3*.tsv"]:
    print("\n", pattern)
    for p in glob.glob(os.path.join(DRIVE_ROOT, pattern), recursive=True):
        print(p)


 **/test_source1*.tsv
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source1.tsv

 **/test_source2*.tsv
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source2.tsv

 **/test_source3*.tsv
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source3.tsv


In [38]:
print("Existing normalization functions:")

for name in [
    "normalize_text",
    "normalize_name",
    "normalize_address",
    "clean_name",
    "clean_address",
    "normalize_country",
    "build_cleaned_source",
]:
    print(name, "->", name in globals())

Existing normalization functions:
normalize_text -> False
normalize_name -> False
normalize_address -> False
clean_name -> False
clean_address -> False
normalize_country -> False
build_cleaned_source -> False


In [39]:
print(s1_blocking.columns.tolist())

['entity_id', 'address_norm', 'country_norm', 'name_core']


In [40]:
print(s1_blocking.head())
print("\nSample normalized names:")
print(s1_blocking["name_core"].head(10).tolist())

print("\nSample normalized addresses:")
print(s1_blocking["address_norm"].head(10).tolist())

print("\nSample countries:")
print(s1_blocking["country_norm"].head(10).tolist())

      entity_id                                      address_norm  \
0  S1-925783039                 1795 westchester dr high point nc   
1  S1-773889195                       17560 ellis rd tahlequah ok   
2  S1-377745466                    1712 montebello ave phoenix az   
3  S1-133037285             2100 cameron dr unit apt g dundalk md   
4  S1-755362802  797 lake town block a kolkata howrah west bengal   

  country_norm                name_core  
0           us      orelee s barbershop  
1           us              prime money  
2           us                 b retail  
3           us            christ chapel  
4        india  prabhav business center  

Sample normalized names:
['orelee s barbershop', 'prime money', 'b retail', 'christ chapel', 'prabhav business center', 'custom wealth services', 'consulting nyasa nursing', 'nexus anchor rain', 'moore bitwise', 'dermatology green medicine']

Sample normalized addresses:
['1795 westchester dr high point nc', '17560 ellis rd tahleq

In [41]:
TEST_S1_PATH = "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source1.tsv"
TEST_S2_PATH = "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source2.tsv"
TEST_S3_PATH = "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source3.tsv"

for path in [TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH]:
    sample = pd.read_csv(path, sep="\t", nrows=2, dtype=str)
    print("\n", os.path.basename(path))
    print(sample.columns.tolist())


 test_source1.tsv
['entity_id', 'business_name', 'business_address', 'country']

 test_source2.tsv
['entity_id', 'business_name', 'business_address', 'country']

 test_source3.tsv
['entity_id', 'business_name', 'business_address', 'country']


In [42]:
for path in [TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH]:

    sample = pd.read_csv(
        path,
        sep="\t",
        nrows=10,
        dtype=str
    )

    print("\n" + "=" * 80)
    print(os.path.basename(path))
    print("=" * 80)

    print(sample[[
        "business_name",
        "business_address",
        "country"
    ]].to_string(index=False))


test_source1.tsv
           business_name                                                                                                                    business_address country
         Zephay Labs Inc                                                                                                         2621 Cotten Road, Tyler, TX      US
    Vision Partners Corp                                                                                              IA, Iowa City, 1064 Newton Rd, Unit 11      US
           << Team Ecole                                                         175 Boulevard du Président Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine  France
     Red Perfect Trading                                                                   Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar, Awas Vikas Colony   India
           ZNB Club SARL                                                                       Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Rue Pierre Dignac 

In [43]:
# Load a small sample of raw TRAIN S1
train_raw_s1 = pd.read_csv(
    os.path.join(
        TRAIN_ROOT,
        "train_source1.tsv"
    ),
    sep="\t",
    nrows=20,
    dtype=str
)

# Load the corresponding cleaned TRAIN S1
train_clean_s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    nrows=20,
    dtype=str
)

print("RAW TRAIN:")
print(
    train_raw_s1[
        ["entity_id", "business_name", "business_address", "country"]
    ].to_string(index=False)
)

print("\n\nCLEANED TRAIN:")
print(
    train_clean_s1.to_string(index=False)
)

RAW TRAIN:
   entity_id                            business_name                                                                                                          business_address country
S1-925783039                      Orelee's Barbershop                                                                                    1795 Westchester Drive, High Point, NC      US
S1-773889195                              Prime Money                                                                                           17560 Ellis Road, Tahlequah, OK      US
S1-377745466                            B+ Retail Inc                                                                                       1712 Montebello Avenue, Phoenix, AZ      US
S1-133037285                            Christ Chapel                                                                         2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US
S1-755362802                  Prabhav Business Center                

In [44]:
import re
import gc
import os
import pandas as pd

# ============================================
# EXACT NORMALIZATION LOGIC FROM 03_Normalization
# ============================================

LEGAL_SUFFIX_MAP = {
    "limited": "ltd",
    "ltd": "ltd",
    "incorporated": "inc",
    "inc": "inc",
    "corporation": "corp",
    "corp": "corp",
    "private": "pvt",
    "pvt": "pvt",
    "company": "co",
    "co": "co",
    "llc": "llc",
    "llp": "llp",
    "plc": "plc"
}

ADDRESS_ABBREVIATIONS = {
    "street": "st",
    "st": "st",
    "road": "rd",
    "rd": "rd",
    "avenue": "ave",
    "ave": "ave",
    "boulevard": "blvd",
    "blvd": "blvd",
    "drive": "dr",
    "dr": "dr",
    "lane": "ln",
    "ln": "ln",
    "parkway": "pkwy",
    "pkwy": "pkwy",
    "highway": "hwy",
    "hwy": "hwy",
    "place": "pl",
    "pl": "pl",
    "court": "ct",
    "ct": "ct",
    "square": "sq",
    "sq": "sq",
    "apartment": "apt",
    "apt": "apt",
    "suite": "ste",
    "ste": "ste",
    "building": "bldg",
    "bldg": "bldg"
}

CORE_SUFFIXES = {
    "ltd",
    "limited",
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "llc",
    "llp",
    "plc",
    "pvt",
    "private",
    "co",
    "company",
    "org",
    "organization"
}


def normalize_basic_series(series):

    return (
        series
        .fillna("")
        .astype("string")
        .str.normalize("NFKC")
        .str.lower()
        .str.replace(
            r"[/\\|,_;:]+",
            " ",
            regex=True
        )
        .str.replace(
            r"[^\w\s]",
            " ",
            regex=True
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
    )


def normalize_name_series(series):

    result = normalize_basic_series(series)

    for old, new in LEGAL_SUFFIX_MAP.items():

        result = result.str.replace(
            rf"\b{re.escape(old)}\b",
            new,
            regex=True
        )

    return result


def normalize_address_series(series):

    result = normalize_basic_series(series)

    for old, new in ADDRESS_ABBREVIATIONS.items():

        result = result.str.replace(
            rf"\b{re.escape(old)}\b",
            new,
            regex=True
        )

    return result


def normalize_country_series(series):

    return (
        series
        .fillna("")
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )


def create_name_core(name_series):

    result = (
        name_series
        .fillna("")
        .astype("string")
        .str.strip()
    )

    for suffix in CORE_SUFFIXES:

        result = result.str.replace(
            rf"\s+\b{re.escape(suffix)}\b$",
            "",
            regex=True
        )

    return (
        result
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [45]:
CLEANED_TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TEST_FILES = [
    ("test_s1", TEST_S1_PATH),
    ("test_s2", TEST_S2_PATH),
    ("test_s3", TEST_S3_PATH)
]

CHUNK_SIZE = 200_000

for name, input_path in TEST_FILES:

    output_path = os.path.join(
        CLEANED_TEST_ROOT,
        f"{name}_cleaned.tsv"
    )

    print("\n" + "=" * 70)
    print("PROCESSING:", name)
    print("=" * 70)

    first_write = True
    total_rows = 0

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        dtype="string",
        chunksize=CHUNK_SIZE
    ):

        # Exact same normalization as 03_Normalization.ipynb
        chunk["name_norm"] = normalize_name_series(
            chunk["business_name"]
        )

        chunk["address_norm"] = normalize_address_series(
            chunk["business_address"]
        )

        chunk["country_norm"] = normalize_country_series(
            chunk["country"]
        )

        chunk["name_missing"] = (
            chunk["name_norm"].str.len() == 0
        ).astype("int8")

        chunk["address_missing"] = (
            chunk["address_norm"].str.len() == 0
        ).astype("int8")

        chunk["name_token_count"] = (
            chunk["name_norm"]
            .str.count(r"\S+")
            .astype("int16")
        )

        chunk["address_token_count"] = (
            chunk["address_norm"]
            .str.count(r"\S+")
            .astype("int16")
        )

        # Exact name_core logic
        chunk["name_core"] = create_name_core(
            chunk["name_norm"]
        )

        chunk.to_csv(
            output_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_rows += len(chunk)

        print(
            f"Processed: {total_rows:,}",
            end="\r"
        )

        del chunk
        gc.collect()

    print(
        f"\nFinished {name}: {total_rows:,} rows"
    )
    print("Saved:", output_path)


PROCESSING: test_s1
Processed: 1,732,544
Finished test_s1: 1,732,544 rows
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/test_s1_cleaned.tsv

PROCESSING: test_s2
Processed: 4,887,273
Finished test_s2: 4,887,273 rows
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/test_s2_cleaned.tsv

PROCESSING: test_s3

Finished test_s3: 5,082,316 rows
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/test_s3_cleaned.tsv


In [46]:
test_check = pd.read_csv(
    os.path.join(
        CLEANED_TEST_ROOT,
        "test_s1_cleaned.tsv"
    ),
    sep="\t",
    nrows=10,
    dtype="string"
)

print(
    test_check[
        [
            "entity_id",
            "business_name",
            "name_norm",
            "name_core",
            "business_address",
            "address_norm",
            "country",
            "country_norm"
        ]
    ].to_string(index=False)
)

   entity_id            business_name                name_norm            name_core                                                                                                                    business_address                                                                                                              address_norm country country_norm
S1-714132312          Zephay Labs Inc          zephay labs inc          zephay labs                                                                                                         2621 Cotten Road, Tyler, TX                                                                                                   2621 cotten rd tyler tx      US           us
S1-106407869     Vision Partners Corp     vision partners corp      vision partners                                                                                              IA, Iowa City, 1064 Newton Rd, Unit 11                                                                   

In [47]:
# ============================================================
# S3 BLOCKING RECALL — EXACT NAME + SELECTIVE TOKEN <= 10
# ============================================================

# We already have:
# s3_name_index
# s3_selective_token_index
# gt_lookup
# s1_blocking

# Collect all TRUE S3 matches
true_s3_total = 0

for matches in gt_lookup.values():
    true_s3_total += sum(
        1 for x in matches
        if x.startswith("S3-")
    )

print("Total true S3 matches:", f"{true_s3_total:,}")

Total true S3 matches: 3,944,746


In [49]:
# ============================================================
# REBUILD TRAIN S3 SELECTIVE TOKEN INDEX
# ============================================================

from collections import defaultdict
import gc

S3_TOKEN_FREQ_MAX = 10
CHUNK_SIZE = 200_000

s3_token_counts = defaultdict(int)

# PASS 1: count token frequencies
for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE
):

    for name in chunk["name_core"]:
        for token in set(meaningful_tokens(name)):
            s3_token_counts[token] += 1

    del chunk
    gc.collect()

print(
    "Unique S3 tokens:",
    f"{len(s3_token_counts):,}"
)

# PASS 2: build selective index
s3_selective_token_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE
):

    for row in chunk.itertuples(index=False):

        entity_id = row.entity_id
        name = row.name_core

        for token in set(meaningful_tokens(name)):

            if s3_token_counts.get(token, 999999) <= S3_TOKEN_FREQ_MAX:
                s3_selective_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print(
    "Selective token keys:",
    f"{len(s3_selective_token_index):,}"
)

print(
    "Indexed S3 references:",
    f"{sum(len(v) for v in s3_selective_token_index.values()):,}"
)

Unique S3 tokens: 862,709
Selective token keys: 804,151
Indexed S3 references: 1,151,548


In [50]:
true_s3_found = 0

for start in range(0, len(s1_blocking), 100_000):

    chunk = s1_blocking.iloc[start:start + 100_000]

    for row in chunk.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core

        true_matches = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith("S3-")
        }

        if not true_matches:
            continue

        candidates = set()

        # Exact country + name
        if pd.notna(country) and pd.notna(name):

            key = f"{country}|{name}"

            candidates.update(
                s3_name_index.get(key, [])
            )

        # Selective token <= 10
        for token in set(meaningful_tokens(name)):

            candidates.update(
                s3_selective_token_index.get(token, [])
            )

        true_s3_found += len(
            true_matches.intersection(candidates)
        )

    print(
        f"Processed {min(start + 100_000, len(s1_blocking)):,} / "
        f"{len(s1_blocking):,}"
    )

print("\n================================")
print("TRUE S3 MATCHES:", f"{true_s3_total:,}")
print("FOUND BY BLOCKER:", f"{true_s3_found:,}")
print(
    "S3 BLOCKING RECALL:",
    f"{100 * true_s3_found / true_s3_total:.4f}%"
)
print("================================")

Processed 100,000 / 2,206,821
Processed 200,000 / 2,206,821
Processed 300,000 / 2,206,821
Processed 400,000 / 2,206,821
Processed 500,000 / 2,206,821
Processed 600,000 / 2,206,821
Processed 700,000 / 2,206,821
Processed 800,000 / 2,206,821
Processed 900,000 / 2,206,821
Processed 1,000,000 / 2,206,821
Processed 1,100,000 / 2,206,821
Processed 1,200,000 / 2,206,821
Processed 1,300,000 / 2,206,821
Processed 1,400,000 / 2,206,821
Processed 1,500,000 / 2,206,821
Processed 1,600,000 / 2,206,821
Processed 1,700,000 / 2,206,821
Processed 1,800,000 / 2,206,821
Processed 1,900,000 / 2,206,821
Processed 2,000,000 / 2,206,821
Processed 2,100,000 / 2,206,821
Processed 2,200,000 / 2,206,821
Processed 2,206,821 / 2,206,821

TRUE S3 MATCHES: 3,944,746
FOUND BY BLOCKER: 180,235
S3 BLOCKING RECALL: 4.5690%


In [ ]:
from collections import defaultdict
import gc

S3_PAIR_CHUNK = 200_000

s3_pair_counts = defaultdict(int)

total = 0

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    dtype=str,
    chunksize=S3_PAIR_CHUNK
):

    for name in chunk["name_core"]:
        pairs = name_token_pairs(name)

        for pair in pairs:
            s3_pair_counts[pair] += 1

    total += len(chunk)

    del chunk
    gc.collect()

print("S3 rows processed:", f"{total:,}")
print("Unique two-token pairs:", f"{len(s3_pair_counts):,}")

In [ ]:
s3_pair_index_5 = defaultdict(list)
s3_pair_index_10 = defaultdict(list)

total = 0

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    dtype=str,
    chunksize=S3_PAIR_CHUNK
):

    for row in chunk.itertuples(index=False):

        entity_id = row.entity_id
        pairs = name_token_pairs(row.name_core)

        for pair in pairs:

            freq = s3_pair_counts.get(pair, 999999)

            if freq <= 5:
                s3_pair_index_5[pair].append(entity_id)

            if freq <= 10:
                s3_pair_index_10[pair].append(entity_id)

    total += len(chunk)

    del chunk
    gc.collect()

print("S3 pair index <=5:")
print("  keys:", f"{len(s3_pair_index_5):,}")
print("  refs:", f"{sum(len(v) for v in s3_pair_index_5.values()):,}")

print("\nS3 pair index <=10:")
print("  keys:", f"{len(s3_pair_index_10):,}")
print("  refs:", f"{sum(len(v) for v in s3_pair_index_10.values()):,}")

In [ ]:
true_s3_total = 0

exact_found = 0
pair5_found = 0
pair10_found = 0

for start in range(0, len(s1_blocking), 100_000):

    chunk = s1_blocking.iloc[start:start + 100_000]

    for row in chunk.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core

        # -----------------------------------------
        # True S3 matches
        # -----------------------------------------

        true_matches = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith("S3-")
        }

        if not true_matches:
            continue

        true_s3_total += len(true_matches)

        # -----------------------------------------
        # Exact country + name candidates
        # -----------------------------------------

        exact_candidates = set()

        if pd.notna(country) and pd.notna(name):

            key = f"{country}|{name}"

            exact_candidates.update(
                s3_name_index.get(key, [])
            )

        exact_found += len(
            true_matches.intersection(exact_candidates)
        )

        # -----------------------------------------
        # Exact + rare pair <=5
        # -----------------------------------------

        pair5_candidates = set(exact_candidates)

        for pair in name_token_pairs(name):

            pair5_candidates.update(
                s3_pair_index_5.get(pair, [])
            )

        pair5_found += len(
            true_matches.intersection(pair5_candidates)
        )

        # -----------------------------------------
        # Exact + rare pair <=10
        # -----------------------------------------

        pair10_candidates = set(exact_candidates)

        for pair in name_token_pairs(name):

            pair10_candidates.update(
                s3_pair_index_10.get(pair, [])
            )

        pair10_found += len(
            true_matches.intersection(pair10_candidates)
        )

    print(
        f"Processed "
        f"{min(start + 100_000, len(s1_blocking)):,} / "
        f"{len(s1_blocking):,}"
    )

print("\n" + "=" * 60)
print("S3 BLOCKING RECALL")
print("=" * 60)

print(
    "Total true S3 matches:",
    f"{true_s3_total:,}"
)

print(
    "\nExact country + name:",
    f"{exact_found:,}",
    f"({100 * exact_found / true_s3_total:.4f}%)"
)

print(
    "\nExact + rare pair <=5:",
    f"{pair5_found:,}",
    f"({100 * pair5_found / true_s3_total:.4f}%)"
)

print(
    "\nExact + rare pair <=10:",
    f"{pair10_found:,}",
    f"({100 * pair10_found / true_s3_total:.4f}%)"
)

print("=" * 60)